# Train Data Analysis — Part 2

## Notebook Overview

In this notebook, we continued the exploratory analysis of the **training dataset** by examining **summary statistics and distributions of clinical and physiological variables**.

The analysis included:

- Summary statistics of **vital signs** such as blood pressure, heart rate, respiratory rate, temperature, and oxygen saturation.
- Examination of **clinical scoring variables**, including **GCS, pain score, and NEWS2**.
- Analysis of **anthropometric measurements** such as **weight, height, and BMI**.
- Review of the **shock index**, a derived indicator of circulatory instability.
- Summary statistics of **emergency department outcomes**, including **disposition** and **ED length of stay**.

This step helps understand **data ranges, variability, potential outliers, and clinical patterns** before moving to deeper analysis and modeling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as pyplot
import seaborn as sns


In [ ]:
df_train = pd.read_csv("../data/train.csv")
df_train.head()

In [ ]:
df_info = pd.DataFrame({
    "Column": df_train.columns,
    "Non-Null Count": df_train.count().values,
    "Null Count": df_train.isnull().sum().values,
    "Data Type": df_train.dtypes.values
})

df_info

In [ ]:
# Identifier columns
id_cols = [
    "patient_id",
    "site_id",
    "triage_nurse_id"
]

# Time-related features
time_cols = [
    "arrival_hour",
    "arrival_day",
    "arrival_month",
    "arrival_season",
    "shift"
]

# Demographic categorical features
demographic_cat = [
    "sex",
    "language",
    "insurance_type",
    "age_group"
]

# Visit context categorical features
context_cat = [
    "arrival_mode",
    "transport_origin",
    "pain_location",
    "mental_status_triage",
    "chief_complaint_system"
]

# Count / utilization features
count_cols = [
    "num_prior_ed_visits_12m",
    "num_prior_admissions_12m",
    "num_active_medications",
    "num_comorbidities"
]

# Continuous vital signs
vital_continuous = [
    "systolic_bp",
    "diastolic_bp",
    "mean_arterial_pressure",
    "pulse_pressure",
    "heart_rate",
    "respiratory_rate",
    "temperature_c",
    "spo2"
]

# Clinical scores (ordinal / discrete)
clinical_scores = [
    "gcs_total",
    "pain_score",
    "news2_score"
]

# Anthropometric continuous features
body_measurements = [
    "weight_kg",
    "height_cm",
    "bmi"
]

# Derived physiological metrics
derived_features = [
    "shock_index"
]

# Outcome / target variables
target_cols = [
    "triage_acuity"
]

# Post-visit outcomes (should not be used as predictors if predicting triage)
post_visit_outcomes = [
    "disposition",
    "ed_los_hours"
]

# `Continuous Features`

In [ ]:
for i in vital_continuous:
    print(f"Summary statistics for {i}:")
    print(df_train[i].describe())
    print("\n") 

## Vital Signs Summary Statistics

This section analyzes the **distribution of key physiological vital signs** recorded at triage.  
Vital signs are critical indicators of **patient stability and immediate clinical risk**, and they play an important role in determining emergency triage severity.

---

## Systolic Blood Pressure (SBP)

| Statistic | Value |
|---|---|
| Mean | 121.61 |
| Median | 123.10 |
| Std | 24.22 |
| Min | 40.00 |
| Max | 226.90 |

**Observations**

- The average SBP (~121 mmHg) falls within the **normal adult range**.
- Very low values (40 mmHg) suggest possible **severe hypotension or measurement anomalies**.
- Extremely high values (>200 mmHg) may indicate **hypertensive emergencies**.

---

## Diastolic Blood Pressure (DBP)

| Statistic | Value |
|---|---|
| Mean | 74.45 |
| Median | 75.30 |
| Std | 14.29 |
| Min | 20.00 |
| Max | 134.80 |

**Observations**

- Mean DBP (~74 mmHg) is within the **normal physiological range**.
- Very low values may reflect **shock or circulatory instability**.
- High values could indicate **severe hypertension**.

---

## Mean Arterial Pressure (MAP)

| Statistic | Value |
|---|---|
| Mean | 90.17 |
| Median | 91.90 |
| Std | 14.17 |
| Min | 30.70 |
| Max | 145.10 |

**Observations**

- MAP around **90 mmHg** suggests generally adequate organ perfusion.
- MAP below **65 mmHg** may indicate **risk of organ hypoperfusion**.

---

## Pulse Pressure

| Statistic | Value |
|---|---|
| Mean | 47.16 |
| Median | 47.20 |
| Std | 24.25 |
| Min | -51.00 |
| Max | 163.70 |

**Observations**

- Typical pulse pressure is around **40–60 mmHg**.
- Negative values likely reflect **measurement inconsistencies or data noise**.
- Very high values may indicate **arterial stiffness or cardiovascular abnormalities**.

---

## Heart Rate

| Statistic | Value |
|---|---|
| Mean | 91.86 |
| Median | 89.60 |
| Std | 19.49 |
| Min | 30.00 |
| Max | 207.70 |

**Observations**

- Mean heart rate (~92 bpm) is slightly elevated compared to typical resting values.
- Extremely low heart rates may represent **bradycardia**.
- Very high values (>200 bpm) may indicate **tachyarrhythmia or severe physiological stress**.

---

## Respiratory Rate

| Statistic | Value |
|---|---|
| Mean | 18.33 |
| Median | 17.30 |
| Std | 4.65 |
| Min | 8.00 |
| Max | 51.50 |

**Observations**

- The average respiratory rate (~18 breaths/min) is within the **normal adult range**.
- High respiratory rates may indicate **respiratory distress or infection**.

---

## Body Temperature

| Statistic | Value |
|---|---|
| Mean | 37.63°C |
| Median | 37.50°C |
| Std | 0.86 |
| Min | 35.10°C |
| Max | 41.80°C |

**Observations**

- Average temperature (~37.6°C) is close to **normal body temperature**.
- High temperatures (>39°C) may indicate **severe infection or systemic inflammation**.

---

## Oxygen Saturation (SpO₂)

| Statistic | Value |
|---|---|
| Mean | 95.79% |
| Median | 97.00% |
| Std | 4.31 |
| Min | 60.40% |
| Max | 100.00% |

**Observations**

- Most patients maintain **adequate oxygen saturation levels**.
- Very low values (<90%) may indicate **respiratory compromise or hypoxia**.

---

## Summary

Overall, the vital signs appear **clinically realistic**, with most values falling within expected physiological ranges.  
However, the presence of **extreme values and missing measurements** suggests potential data variability that may require **cleaning or outlier handling during preprocessing**.



`  min pulse_pressure = -51    A small number of physiologically implausible values are observed, particularly negative pulse pressure values, which likely result from measurement or calculation inconsistencies. Additionally, several extreme vital sign values appear that may represent either severe clinical conditions or measurement noise.`

# `Clinical Scores Features`

In [ ]:
for i in clinical_scores:
    print(f"Summary statistics for {i}:")
    print(df_train[i].describe())
    print("\n")

## Clinical Assessment Scores

This section summarizes key **clinical scoring systems used during triage**.  
These scores help clinicians rapidly assess **neurological status, pain severity, and physiological deterioration risk**.

---

## Glasgow Coma Scale (GCS)

| Statistic | Value |
|---|---|
| Mean | 14.15 |
| Median | 15 |
| Std | 2.15 |
| Min | 3 |
| Max | 15 |

**Observations**

- Most patients have **GCS = 15**, indicating **normal consciousness**.
- Lower values indicate **impaired neurological status**.
- The minimum score of **3** corresponds to **deep unconsciousness or coma**.

---

## Pain Score

| Statistic | Value |
|---|---|
| Mean | 4.51 |
| Median | 5 |
| Std | 3.36 |
| Min | -1 |
| Max | 10 |

**Observations**

- Pain scores range from **0–10**, representing increasing pain intensity.
- The average score (~4.5) suggests **moderate pain levels** in many patients.
- The value **-1 likely indicates missing or unrecorded pain scores**.

---

## NEWS2 Score

| Statistic | Value |
|---|---|
| Mean | 3.43 |
| Median | 2 |
| Std | 4.26 |
| Min | 0 |
| Max | 17 |

**Observations**

- NEWS2 measures **physiological deterioration risk** based on vital signs.
- Most patients have **low scores**, indicating **low immediate risk**.
- Higher scores (≥7) may indicate **serious clinical deterioration requiring urgent care**.

---

## Summary

These clinical scores capture **critical patient status during triage**:

- **GCS** reflects neurological consciousness.
- **Pain score** indicates patient discomfort and symptom severity.
- **NEWS2 score** summarizes physiological instability.

Together, these features provide strong signals for **predicting triage acuity and clinical severity** in emergency department settings.

`Note pain_score = -1 → missing indicator gcs_total highly skewed (most = 15) very high NEWS2 values may indicate severe cases`

# `Body Measurements Features`


In [ ]:
for i in body_measurements:
    print(f"Summary statistics for {i}:")
    print(df_train[i].describe())
    print("\n")

## Anthropometric Measurements

This section summarizes **body measurement features** including weight, height, and Body Mass Index (BMI).  
These variables provide insight into **patient body composition and nutritional status**, which may influence disease risk and clinical outcomes in emergency care.

---

## Weight (kg)

| Statistic | Value |
|---|---|
| Mean | 74.45 kg |
| Median | 76.00 kg |
| Std | 21.34 |
| Min | 2.00 kg |
| Max | 148.50 kg |

**Observations**

- The average weight (~74 kg) is within a typical adult range.
- The minimum value of **2 kg likely represents pediatric patients or potential data entry anomalies**.
- Very high weights (>120 kg) indicate **severe obesity in a subset of patients**.

---

## Height (cm)

| Statistic | Value |
|---|---|
| Mean | 168.62 cm |
| Median | 171.10 cm |
| Std | 16.59 |
| Min | 45.00 cm |
| Max | 210.00 cm |

**Observations**

- Average height (~169 cm) aligns with typical adult populations.
- Extremely low values such as **45 cm likely correspond to pediatric patients**.
- Maximum values (~210 cm) represent very tall individuals but remain within possible biological limits.

---

## Body Mass Index (BMI)

| Statistic | Value |
|---|---|
| Mean | 26.36 |
| Median | 26.00 |
| Std | 7.67 |
| Min | 10.00 |
| Max | 65.00 |

**Observations**

- The average BMI (~26) falls within the **overweight category**.
- BMI values below **18.5 indicate underweight patients**, which may be associated with malnutrition or chronic illness.
- Very high BMI values (>40) suggest **severe obesity**, which increases risk for cardiovascular and metabolic complications.

---

## Summary

Anthropometric features reveal **wide variation in patient body composition**, reflecting the presence of both **pediatric and adult populations** as well as individuals across different weight categories.

These variables may influence **disease risk, medication dosing, and physiological stress responses**, making them useful contextual features for **triage severity prediction**.


`### ⚠️ Potential Red Flags in Anthropometric Data`

| Feature | Value | Note |
|---|---|---|
| **weight_kg (min)** | 2 kg | Extremely low; likely newborn/pediatric entry or possible data entry error. |
| **height_cm (min)** | 45 cm | Very short height, consistent with infants but unusual for general ED population. |
| **bmi (min)** | 10 | Indicates severe underweight; could represent malnutrition, pediatric case, or measurement issue. |
| **bmi (max)** | 65 | Extremely high BMI indicating severe morbid obesity. |

**Conclusion:**  
The dataset likely contains **both pediatric and adult patients**, which explains extreme values in weight, height, and BMI. However, these outliers should be **validated or handled carefully during preprocessing** to avoid distortion in model training.

# `Derived Features`


In [ ]:
df_train[derived_features].describe()

## Shock Index (SI)

| Statistic | Value |
|---|---|
| Count | 75,854 |
| Mean | 0.81 |
| Std | 0.33 |
| Min | 0.19 |
| 25% | 0.60 |
| Median | 0.72 |
| 75% | 0.92 |
| Max | 4.77 |

### Clinical Context


It is used to assess **hemodynamic instability and risk of shock**.

Typical interpretation:

| SI Range | Interpretation |
|---|---|
| 0.5 – 0.7 | Normal |
| 0.7 – 0.9 | Mild physiological stress |
| > 1.0 | Possible shock / circulatory compromise |
| > 1.3 | High risk of severe shock |

### ⚠️ Potential Red Flags

| Observation | Value | Note |
|---|---|---|
| Very high SI | 4.77 | Extremely abnormal; may indicate severe shock, extreme vitals, or calculation error. |
| High upper quartile | 0.92 | Indicates a notable portion of patients approaching hemodynamic instability. |

**Conclusion**

Most patients fall within **normal to mildly elevated shock index ranges**, but extreme values suggest **possible severe circulatory compromise or measurement anomalies**, which should be carefully inspected during preprocessing.

# Post Visit Outcomes Features

In [ ]:
for i in post_visit_outcomes:
    print(f"Summary statistics for {i}:")
    print(df_train[i].describe())
    print("\n")

In [ ]:
for i in post_visit_outcomes:
    print(f"Summary statistics for {i}:")
    print(df_train[i].value_counts())
    print("\n")

## Emergency Department Outcomes

This section summarizes **post-visit outcomes** including patient disposition and emergency department length of stay (ED LOS).

---

## Disposition

| Outcome | Count |
|---|---|
| discharged | 39,028 |
| admitted | 24,601 |
| transferred | 5,203 |
| observation | 4,337 |
| lwbs (left without being seen) | 3,656 |
| lama (left against medical advice) | 2,764 |
| deceased | 411 |

### Observations

- Nearly **49% of patients were discharged**, indicating many visits were non-critical.
- About **31% required hospital admission**, suggesting a substantial number of moderate to severe cases.
- A small number of patients **left before treatment or against medical advice**.
- **Deaths are rare (~0.5%)**, which is expected in general emergency department data.

---

## Emergency Department Length of Stay (ED LOS)

| Statistic | Value |
|---|---|
| Mean | 3.50 hours |
| Median | 3.00 hours |
| Std | 2.44 |
| Min | 0.00 |
| Max | 17.51 |

### Observations

- The median stay of **~3 hours** suggests typical emergency department throughput.
- The maximum stay of **17.5 hours** indicates some patients required extended observation or complex care.
- A LOS of **0 hours may represent immediate transfer, rapid discharge, or data rounding**.

---

### ⚠️ Modeling Note

Both **`disposition`** and **`ed_los_hours`** occur **after triage**, meaning they are **post-outcome variables**.

Using them as predictors for **triage_acuity** would introduce **data leakage**, since they contain information about events that happen after the triage decision.

Therefore, these variables should **not be used as input features during model training**.